In [7]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
df_full = pd.read_csv("005_manchester_processed.csv")
print(df_full.isna().sum())

# sort
df = df_full.sort_values(by='time')

df = df_full.drop(columns=['lat', 'lon', 'time', 'date'], errors='ignore')


time          0
TREFMXAV_U    0
FLNS          0
FSNS          0
PRECT         0
PRSN          0
QBOT          0
TREFHT        0
UBOT          0
VBOT          0
lat           0
lon           0
year          0
month         0
day           0
dayofyear     0
dtype: int64


In [12]:
#dataset split
train_df = df[(df['year'] >= 2006) & (df['year'] <= 2040)]
valid_df = df[(df['year'] > 2040) & (df['year'] < 2050)]
test_df  = df[(df['year'] >= 2050) & (df['year'] <= 2080)]

target = 'TREFMXAV_U'
features = [col for col in train_df.columns if col != target]

X_train, y_train = train_df[features], train_df[target]
X_valid, y_valid = valid_df[features], valid_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# validation
y_valid_pred = rf.predict(X_valid)

val_mae = mean_absolute_error(y_valid, y_valid_pred)
val_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred))

print("===== Validation Results =====")
print("MAE:", val_mae)
print("RMSE:", val_rmse)

===== Validation Results =====
MAE: 0.5513654399195387
RMSE: 0.7324847354458368


In [14]:
from sklearn.metrics import r2_score, max_error

y_test_pred = rf.predict(X_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)
test_max_err = max_error(y_test, y_test_pred)

print("\n===== Test Results =====")
print("MAE:", test_mae)
print("RMSE:", test_rmse)
print("R²:", test_r2)
print("Max Error:", test_max_err)


===== Test Results =====
MAE: 0.5612539201530198
RMSE: 0.7372941628502372
R²: 0.9809608116531741
Max Error: 4.84361122809524


In [15]:
from xgboost import XGBRegressor
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# training
xgb.fit(X_train, y_train)


y_valid_pred_xgb = xgb.predict(X_valid)

xgb_val_mae = mean_absolute_error(y_valid, y_valid_pred_xgb)
xgb_val_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred_xgb))

print("\n===== XGBoost Validation =====")
print("MAE:", xgb_val_mae)
print("RMSE:", xgb_val_rmse)




===== XGBoost Validation =====
MAE: 0.518153575307479
RMSE: 0.7087473979589844


In [16]:
# final training
X_final = pd.concat([X_train, X_valid])
y_final = pd.concat([y_train, y_valid])

xgb.fit(X_final, y_final)

# test
y_test_pred_xgb = xgb.predict(X_test)

xgb_test_mae = mean_absolute_error(y_test, y_test_pred_xgb)
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_xgb))
xgb_test_r2 = r2_score(y_test, y_test_pred_xgb)
xgb_test_max_err = max_error(y_test, y_test_pred_xgb)

print("\n===== XGBoost Test Results =====")
print("MAE:", xgb_test_mae)
print("RMSE:", xgb_test_rmse)
print("R²:", xgb_test_r2)
print("Max Error:", xgb_test_max_err)


===== XGBoost Test Results =====
MAE: 0.5174809982446376
RMSE: 0.6881523534638848
R²: 0.9834142147488408
Max Error: 4.689192579345729


In [17]:
from lightgbm import LGBMRegressor

# LGBM model
lgb = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# training
lgb.fit(X_train, y_train)

# Validation
y_valid_pred_lgb = lgb.predict(X_valid)

lgb_val_mae = mean_absolute_error(y_valid, y_valid_pred_lgb)
lgb_val_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred_lgb))

print("\n===== LightGBM Validation =====")
print("MAE:", lgb_val_mae)
print("RMSE:", lgb_val_rmse)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001000 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2579
[LightGBM] [Info] Number of data points in the train set: 12311, number of used features: 12
[LightGBM] [Info] Start training from score 14.801742

===== LightGBM Validation =====
MAE: 0.5186234503185531
RMSE: 0.7039469039089346


In [18]:
# final training
lgb.fit(X_final, y_final)

# test
y_test_pred_lgb = lgb.predict(X_test)

lgb_test_mae = mean_absolute_error(y_test, y_test_pred_lgb)
lgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_lgb))
lgb_test_r2 = r2_score(y_test, y_test_pred_lgb)
lgb_test_max_err = max_error(y_test, y_test_pred_lgb)

print("\n===== LightGBM Test Results =====")
print("MAE:", lgb_test_mae)
print("RMSE:", lgb_test_rmse)
print("R²:", lgb_test_r2)
print("Max Error:", lgb_test_max_err)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000319 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2592
[LightGBM] [Info] Number of data points in the train set: 15457, number of used features: 12
[LightGBM] [Info] Start training from score 15.018299

===== LightGBM Test Results =====
MAE: 0.5181057602123259
RMSE: 0.6838496686115392
R²: 0.9836209721801552
Max Error: 4.25088063704497
